In [ ]:
"""
=============================================================
FILE 35 — MEMORY AUGMENTED AGENTS
=============================================================

CONCEPTS TAUGHT
----------------
1. Memory-Augmented Agents
2. Persistent Memory
3. Conversational Memory
4. Stateful Agents
5. Personalized AI
6. Long-Term Context
7. Session Continuity
8. Agent Recall
9. Memory Systems
10. Context Retention

CORE IDEA
-----------
Agents remember previous interactions.

FLOW
-----
Conversation
   ↓
Store Memory
   ↓
Retrieve Memory
   ↓
Personalized Response

REAL WORLD USE CASES
---------------------
- AI copilots
- CRM assistants
- Personalized AI
- Enterprise assistants
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict
from typing import Annotated

from langgraph.graph.message import add_messages

from langgraph.graph import StateGraph, START, END

from langgraph.checkpoint.memory import MemorySaver

from IPython.display import Image, display

# ============================================================
# STEP 2 — LOAD ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):
    messages: Annotated[list, add_messages]

# ============================================================
# STEP 5 — MEMORY AGENT
# ============================================================

def memory_agent(state: State):

    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }

# ============================================================
# STEP 6 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node(
    "memory_agent",
    memory_agent
)

builder.add_edge(
    START,
    "memory_agent"
)

builder.add_edge(
    "memory_agent",
    END
)

# ============================================================
# STEP 7 — MEMORY CHECKPOINTER
# ============================================================

memory = MemorySaver()

# ============================================================
# STEP 8 — COMPILE GRAPH
# ============================================================

graph = builder.compile(
    checkpointer=memory
)

# ============================================================
# STEP 9 — VISUALIZE GRAPH
# ============================================================

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 10 — THREAD CONFIG
# ============================================================

config = {
    "configurable": {
        "thread_id": "user_123"
    }
}

# ============================================================
# STEP 11 — FIRST MESSAGE
# ============================================================

result_1 = graph.invoke(
    {
        "messages": [
            (
                "human",
                "My name is Utsab and I work in AI."
            )
        ]
    },
    config=config
)

# ============================================================
# STEP 12 — SECOND MESSAGE
# ============================================================

result_2 = graph.invoke(
    {
        "messages": [
            (
                "human",
                "What do you remember about me?"
            )
        ]
    },
    config=config
)

# ============================================================
# STEP 13 — PRINT RESULTS
# ============================================================

print("\nMEMORY RESPONSE\n")
print("=" * 60)

for msg in result_2["messages"]:
    print(msg)